In [1]:
import pandas as pd

path = "../outputs/lactate_extraction_results.csv"

In [2]:
df = pd.read_csv(path)
#df = df.loc[df['n_chunks'] >= 1, :]

discharge = pd.read_csv("../../Outputs/discharge_filtered.csv")

In [3]:
#df.head()

In [4]:
import pandas as pd
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score
)

eval_df = df.copy()

eval_df = eval_df[eval_df["llm_present"].notna() & eval_df["structured_lactate"].notna()].copy()

eval_df["pred_present"] = eval_df["llm_present"].astype(bool)
eval_df["gold_present"] = eval_df["structured_lactate"] > 2.0

print("N =", len(eval_df))
print("Accuracy:", accuracy_score(eval_df["gold_present"], eval_df["pred_present"]))
print("Precision:", precision_score(eval_df["gold_present"], eval_df["pred_present"], zero_division=0))
print("Recall:", recall_score(eval_df["gold_present"], eval_df["pred_present"], zero_division=0))
print("F1:", f1_score(eval_df["gold_present"], eval_df["pred_present"], zero_division=0))

print("\nConfusion matrix:")
print(confusion_matrix(eval_df["gold_present"], eval_df["pred_present"]))

print("\nClassification report:")
print(classification_report(eval_df["gold_present"], eval_df["pred_present"], zero_division=0))

N = 927
Accuracy: 0.7281553398058253
Precision: 0.7693430656934307
Recall: 0.8486312399355878
F1: 0.8070444104134763

Confusion matrix:
[[148 158]
 [ 94 527]]

Classification report:
              precision    recall  f1-score   support

       False       0.61      0.48      0.54       306
        True       0.77      0.85      0.81       621

    accuracy                           0.73       927
   macro avg       0.69      0.67      0.67       927
weighted avg       0.72      0.73      0.72       927



In [5]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

value_df = df.copy()
value_df = value_df[
    value_df["llm_lactate_value"].notna() &
    value_df["structured_lactate"].notna()
].copy()

print("N with comparable values =", len(value_df))
print("MAE:", mean_absolute_error(value_df["structured_lactate"], value_df["llm_lactate_value"]))
print("RMSE:", np.sqrt(mean_squared_error(value_df["structured_lactate"], value_df["llm_lactate_value"])))
print("Correlation:", value_df["structured_lactate"].corr(value_df["llm_lactate_value"]))

value_df["abs_error"] = (value_df["structured_lactate"] - value_df["llm_lactate_value"]).abs()
value_df["signed_error"] = value_df["llm_lactate_value"] - value_df["structured_lactate"]

N with comparable values = 957
MAE: 2.849903294439171
RMSE: 4.114895210325117
Correlation: 0.5761167298356504


In [6]:
def lactate_category(x):
    if pd.isna(x):
        return np.nan
    elif x <= 2:
        return "normal"
    elif x < 4:
        return "mildly_elevated"
    else:
        return "severely_elevated"

cat_df = df.copy()
cat_df = cat_df[
    cat_df["llm_lactate_value"].notna() &
    cat_df["structured_lactate"].notna()
].copy()

cat_df["gold_cat"] = cat_df["structured_lactate"].apply(lactate_category)
cat_df["pred_cat"] = cat_df["llm_lactate_value"].apply(lactate_category)

agreement = (cat_df["gold_cat"] == cat_df["pred_cat"]).mean()
print("Category agreement:", agreement)

pd.crosstab(cat_df["gold_cat"], cat_df["pred_cat"], margins=True)

Category agreement: 0.5182863113897597


pred_cat,mildly_elevated,normal,severely_elevated,All
gold_cat,,,,
mildly_elevated,108,36,159,303
normal,119,115,81,315
severely_elevated,39,27,273,339
All,266,178,513,957


# Main Analysis

In [7]:
import pandas as pd
import numpy as np
import re
import ast
from sklearn.metrics import classification_report, confusion_matrix

def normalize_chunks(chunks):
    if pd.isna(chunks):
        return ""
    if isinstance(chunks, str):
        try:
            parsed = ast.literal_eval(chunks)
            if isinstance(parsed, list):
                return " ".join(str(x) for x in parsed)
        except:
            pass
        return chunks
    if isinstance(chunks, list):
        return " ".join(str(x) for x in chunks)
    return str(chunks)

def extract_lactate_values(text):
    text = str(text).lower()

    pattern = r"(?:lactate|lactic acid)\s*(?:was|is|of|:|=|-)?\s*([0-9]+(?:\.[0-9]+)?)"

    values = []
    for m in re.findall(pattern, text):
        try:
            values.append(float(m))
        except:
            pass

    return values

def derive_chunk_gold_label(text):
    t = str(text).lower().strip()
    if not t:
        return None
    
    if not re.search(r"\b(lactate|lactic acid|lactic acidosis)\b", t):
        return None

    if re.search(r"lactic acidosis|elevated lactate|high lactate|lactate (?:is |was )?(?:elevated|high)", t):
        return True

    if re.search(r"normal lactate|lactate (?:is |was )?(?:normal|cleared|normalized|not elevated)|lactate cleared|lactate normalized", t):
        return False

    values = extract_lactate_values(t)
    if values:
        if any(v > 2.0 for v in values):
            return True
        if all(v <= 2.0 for v in values):
            return False

    return None

def normalize_llm_present(x):
    if pd.isna(x):
        return None
    if x is True or x is False:
        return x
    if isinstance(x, str):
        x = x.strip().lower()
        if x == "true":
            return True
        if x == "false":
            return False
    return None

df["chunk_text"] = df["chunks"].apply(normalize_chunks)
df["gold_present_from_chunks"] = df["chunk_text"].apply(derive_chunk_gold_label)
df["llm_present_norm"] = df["llm_present"].apply(normalize_llm_present)

In [8]:
# Mention detection
df["gold_mentioned"] = df["chunk_text"].str.lower().str.contains(
    r"\b(?:lactate|lactic acid|lactic acidosis)\b",
    regex=True,
    na=False
)
df["llm_mentioned"] = df["llm_present_norm"].notna()

print("=== Mention detection ===")
print(classification_report(df["gold_mentioned"], df["llm_mentioned"], zero_division=0))

=== Mention detection ===
              precision    recall  f1-score   support

       False       0.83      1.00      0.91       577
        True       1.00      0.89      0.94      1041

    accuracy                           0.93      1618
   macro avg       0.91      0.94      0.92      1618
weighted avg       0.94      0.93      0.93      1618



In [9]:
print("=== Elevated detection ===")
compare_df = df[
    df["gold_present_from_chunks"].notna() &
    df["llm_present_norm"].notna()
].copy()

y_true = compare_df["gold_present_from_chunks"].astype(bool)
y_pred = compare_df["llm_present_norm"].astype(bool)

print("=== Elevated detection ===")
print(classification_report(y_true, y_pred, zero_division=0))
print(confusion_matrix(y_true, y_pred))

=== Elevated detection ===
=== Elevated detection ===
              precision    recall  f1-score   support

       False       0.62      0.94      0.75       158
        True       0.99      0.88      0.93       753

    accuracy                           0.89       911
   macro avg       0.81      0.91      0.84       911
weighted avg       0.92      0.89      0.90       911

[[149   9]
 [ 90 663]]


In [10]:
mismatch_df = compare_df[
    compare_df["gold_present_from_chunks"] != compare_df["llm_present_norm"]
]

mismatchs = mismatch_df.head(20)[["chunk_text","gold_present_from_chunks","llm_present_norm","llm_evidence_quote"]]

In [11]:
def return_text_llm_answer_gold_answer(df):
    for idx, row in df.iterrows():
        print(row['chunk_text'])
        print('gold: ', row['gold_present_from_chunks'])
        print('llm_present: ', row['llm_present_norm'])
        print('llm_evidence_quote: ', row['llm_evidence_quote'])
        print('*******************************************************')

In [12]:
#return_text_llm_answer_gold_answer(mismatchs)

In [13]:
df["llm_elevated_from_value"] = df["llm_lactate_value"] > 2

compare_df = df[
    df["gold_present_from_chunks"].notna() &
    df["llm_lactate_value"].notna()
]

y_true = compare_df["gold_present_from_chunks"].astype(bool)
y_pred = compare_df["llm_elevated_from_value"].astype(bool)

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

       False       0.90      0.95      0.92       169
        True       0.99      0.98      0.98       775

    accuracy                           0.97       944
   macro avg       0.95      0.96      0.95       944
weighted avg       0.97      0.97      0.97       944



In [ ]:
mismatch_df = compare_df[
    compare_df["gold_present_from_chunks"] != compare_df["llm_elevated_from_value"]
]

mismatchs = mismatch_df.head(20)[["chunk_text","gold_present_from_chunks","llm_elevated_from_value","llm_evidence_quote"]]

In [15]:
#mismatchs

In [16]:
#for idx, row in mismatchs.iterrows():
#        print(row['chunk_text'])
#        print('gold: ', row['gold_present_from_chunks'])
#        print('llm_present: ', row['llm_elevated_from_value'])
#        print('llm_evidence_quote: ', row['llm_evidence_quote'])
#        print('*******************************************************')